In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer

In [33]:
%run ../NewtonIter_GenerateSamples.py -N 20000 -v

Summary stats
	mean delta T: 0.3067042543365101
	variance in delta T: 7.382655575512571
	max delta T: 139.5374953986583
	min delta T: -61.35332898781462
	min |delta T|: 7.010129365880857e-09 

	mean delta qv: -0.00012323376939240972
	variance in delta qv: 1.191879835914814e-06
	max delta qv: 1.191879835914814e-06
	min delta qv: -0.05606616565118091
	min |delta qv|: 2.816666196499518e-12 

	mean delta qc: 0.00012323376939240972
	variance in delta qc: 1.191879835914814e-06
	max delta qc: 0.0560661656511809
	min delta qc: -0.024651767587303905
	min |delta qc|: 2.8166661711348517e-12


In [34]:
df = pd.read_csv("../newton_samples.csv")

In [35]:
X = df[['T_in', 'pres_in', 'qv_in', 'qc_in']].values
Y = df[['T_out', 'qv_out', 'qc_out']].values

In [36]:
scaler_x = QuantileTransformer()
scaler_y = QuantileTransformer()

In [37]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3)

In [38]:
X_train_scaled = torch.tensor(scaler_x.fit_transform(X_train), dtype=torch.float64)
X_test_scaled = torch.tensor(scaler_x.fit_transform(X_test), dtype=torch.float64)
Y_train_scaled = torch.tensor(scaler_x.fit_transform(Y_train), dtype=torch.float64)
Y_test_scaled = torch.tensor(scaler_x.fit_transform(Y_test), dtype=torch.float64)

In [39]:
X_train_scaled

tensor([[0.9186, 0.6400, 0.8611, 0.2201],
        [0.9332, 0.9763, 0.8905, 0.2943],
        [0.7923, 0.8658, 0.7296, 0.8871],
        ...,
        [0.4896, 0.4254, 0.5515, 0.3944],
        [0.5581, 0.5124, 0.6106, 0.9269],
        [0.1665, 0.4264, 0.1679, 0.6623]], dtype=torch.float64)

In [40]:
X_train

array([[3.12659471e+02, 6.47831377e+04, 4.91945684e-04, 1.09115990e-03],
       [3.14532691e+02, 9.78256617e+04, 6.46837042e-04, 1.46621309e-03],
       [2.95864671e+02, 8.74632781e+04, 1.42836292e-04, 4.43322990e-03],
       ...,
       [2.57046950e+02, 4.28219302e+04, 2.69607250e-05, 1.95102832e-03],
       [2.65292344e+02, 5.10229435e+04, 4.89799552e-05, 4.61623007e-03],
       [2.13203990e+02, 4.29117285e+04, 1.57799917e-07, 3.33595294e-03]],
      shape=(7000, 4))

In [41]:
Y_train

array([[3.12164971e+02, 6.90636018e-04, 8.92469562e-04],
       [3.14885502e+02, 5.05077483e-04, 1.60797265e-03],
       [2.95731568e+02, 1.96317028e-04, 4.37974917e-03],
       ...,
       [2.57050952e+02, 2.53527113e-05, 1.95263633e-03],
       [2.65311516e+02, 4.12764054e-05, 4.62393362e-03],
       [2.13203681e+02, 2.81891068e-07, 3.33582885e-03]], shape=(7000, 3))

In [59]:
class NewtonNet(nn.Module):
    def __init__(self, n_in, n_out, width):
        super(NewtonNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(n_in, width),
            nn.ReLU(),
            nn.Linear(width, width),
            nn.ReLU(),
            nn.Linear(width, n_out)
        )

    def forward(self, X):
        return self.model(X)

In [60]:
n_in = 4
n_out = 3
width = 64
m_tol = 1e-8

In [61]:
net = NewtonNet(n_in, n_out, width)
net.double()
criterion = nn.MSELoss()
optimizer = optim.Adam(net.parameters())

In [62]:
epochs = 10000
for epoch in range(epochs):
    net.train()
    
    pred = net(X_train_scaled)
    loss = criterion(pred, Y_train_scaled)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.3e}")

    if (loss.item() < m_tol) :
        print(f"Breaking at Epoch {epoch}, Loss: {loss.item():.3e}")
        break

Epoch 0, Loss: 3.256e-01
Epoch 100, Loss: 3.559e-03
Epoch 200, Loss: 2.311e-03
Epoch 300, Loss: 2.067e-03
Epoch 400, Loss: 1.915e-03
Epoch 500, Loss: 1.836e-03
Epoch 600, Loss: 1.782e-03
Epoch 700, Loss: 1.738e-03
Epoch 800, Loss: 1.700e-03
Epoch 900, Loss: 1.665e-03
Epoch 1000, Loss: 1.630e-03
Epoch 1100, Loss: 1.593e-03
Epoch 1200, Loss: 1.556e-03
Epoch 1300, Loss: 1.522e-03
Epoch 1400, Loss: 1.489e-03
Epoch 1500, Loss: 1.458e-03
Epoch 1600, Loss: 1.427e-03
Epoch 1700, Loss: 1.394e-03
Epoch 1800, Loss: 1.355e-03
Epoch 1900, Loss: 1.311e-03
Epoch 2000, Loss: 1.271e-03
Epoch 2100, Loss: 1.232e-03
Epoch 2200, Loss: 1.196e-03
Epoch 2300, Loss: 1.160e-03
Epoch 2400, Loss: 1.131e-03
Epoch 2500, Loss: 1.089e-03
Epoch 2600, Loss: 1.050e-03
Epoch 2700, Loss: 1.023e-03
Epoch 2800, Loss: 9.950e-04
Epoch 2900, Loss: 9.646e-04
Epoch 3000, Loss: 9.375e-04
Epoch 3100, Loss: 9.153e-04
Epoch 3200, Loss: 8.804e-04
Epoch 3300, Loss: 8.737e-04
Epoch 3400, Loss: 8.192e-04
Epoch 3500, Loss: 7.977e-04
Epoc

In [63]:
net.eval()
with torch.no_grad():
    Y_pred_scaled = net(X_test_scaled)
    
    loss_T = criterion(Y_pred_scaled[0], Y_test_scaled[0])
    print(loss_T)

    loss_qv = criterion(Y_pred_scaled[1], Y_test_scaled[1])
    print(loss_qv)

    loss_qc = criterion(Y_pred_scaled[2], Y_test_scaled[2])
    print(loss_qc)

tensor(5.7448e-05, dtype=torch.float64)
tensor(7.8212e-05, dtype=torch.float64)
tensor(4.9638e-05, dtype=torch.float64)
